# 12 — Multivariate forcing: source audit

Audits every candidate exogenous variable before any of it reaches a model:
temporal resolution, units, grid orientation, missingness, and provenance.

The SST-only experiment in notebooks `00`–`11` remains the interpretable
baseline. This is a **separate question**: does exogenous information add
forecast skill beyond what SST history alone supports?

## Two sources, very different costs

| Source | Variables | Access |
|---|---|---|
| NOAA OISST | `err`, `ice`, `anom` | Same ERDDAP request as `sst` — no credentials, already on the study grid |
| ERA5 (Copernicus C3S) | winds, air temperature, pressure, heat fluxes | Free account **and a personal access token** required |

So the OISST auxiliary channels are runnable immediately; the ERA5 arm needs
credentials this notebook deliberately does not assume.

In [ ]:
from pathlib import Path

import numpy as np

from oisst_fno.data import Region, download_subset, open_oisst
from oisst_fno.multivariate import (
    ALIGNMENT_DECISIONS,
    ERA5_DOI,
    ERA5_LATENCY_DAYS,
    ERA5_LICENSE,
    ERA5_PROVIDER,
    ERA5_VARIABLES,
    OISST_AUX_VARIABLES,
    build_era5_request,
    era5_area_from_region,
)
from oisst_fno.validation import validate_oisst_dataset

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
REGION = Region()

print("Alignment decisions in force:")
for index, decision in enumerate(ALIGNMENT_DECISIONS, start=1):
    print(f"  {index}. {decision}")

## Candidate variables

Each carries its own regridding method and daily reduction, because the right
choice differs: accumulated fluxes must be summed rather than averaged, and sea
ice must not be smeared across its boundary by bilinear interpolation.

In [ ]:
print(f"{'name':>6}  {'source':>6}  {'units':>8}  {'regrid':>8}  {'daily':>8}  description")
for spec in list(OISST_AUX_VARIABLES.values()) + list(ERA5_VARIABLES.values()):
    print(
        f"{spec.name:>6}  {spec.source:>6}  {spec.units:>8}  "
        f"{spec.regrid_method:>8}  {spec.daily_reduction:>8}  {spec.description}"
    )

## OISST auxiliary fields

Downloaded alongside `sst` in one request, so they inherit its grid, its time
axis, and its provenance manifest.

In [ ]:
# The auxiliary fields come from the same ERDDAP request as sst: no credentials needed.
START_DATE = "2020-03-01"
END_DATE = "2026-07-31"
AUX_PATH = RAW / f"oisst_aux_{START_DATE}_{END_DATE}_ne_atlantic.nc"

aux_path = download_subset(
    destination=AUX_PATH,
    start_date=START_DATE,
    end_date=END_DATE,
    region=REGION,
    variables=("sst", "err", "ice"),
)
aux = open_oisst(aux_path)
print(aux)

### Units, ranges, missingness, and mask agreement

In [ ]:
report = validate_oisst_dataset(aux)
for key, value in report.summary.items():
    print(f"{key:>12}: {value}")
for issue in report.issues:
    print(f"ISSUE {issue}")
report.raise_for_status()

# Per-variable audit: units, range, missingness, and whether the mask matches sst.
sst_missing = ~np.isfinite(aux["sst"].values)
print(f"\n{'var':>5} {'min':>10} {'max':>10} {'mean':>10} {'missing %':>10} {'mask == sst':>12}")
for name in ("sst", "err", "ice"):
    values = aux[name].values
    finite = np.isfinite(values)
    missing = ~finite
    same_mask = bool(np.array_equal(missing, sst_missing))
    print(
        f"{name:>5} {np.nanmin(values):>10.3f} {np.nanmax(values):>10.3f} "
        f"{np.nanmean(values):>10.3f} {100 * missing.mean():>10.2f} {str(same_mask):>12}"
    )

### Usability verdict

Running this audit over the Northeast Atlantic domain turns up a concrete result:
**`ice` is 100% missing** across 30–50°N, because there is no sea ice at these
latitudes. It is a valid OISST variable and a reasonable candidate in general, but
it carries no information *here*.

This is precisely what the audit exists to catch. Without it, `ice` would reach
`PerVariableStandardizer.fit`, which raises on a non-finite statistic — a correct
but much more confusing failure, several notebooks later.

In [ ]:
# A channel that is entirely missing, or constant, over the study region carries no
# information and must not reach a model. Detected here rather than discovered later as
# a confusing training failure.
USABLE = {}
for name in ("sst", "err", "ice"):
    values = aux[name].values
    finite = np.isfinite(values)
    if not finite.any():
        verdict, why = False, "entirely missing over this region"
    elif float(np.nanstd(values)) == 0.0:
        verdict, why = False, "constant, so it cannot be standardized"
    else:
        verdict, why = True, f"std {float(np.nanstd(values)):.4f} {name != 'ice' and 'C' or 'fraction'}"
    USABLE[name] = verdict
    print(f"{name:>5}: {'USABLE  ' if verdict else 'UNUSABLE'}  {why}")

unusable = [name for name, ok in USABLE.items() if not ok]
if unusable:
    print(f"\nExcluded from the experiment in notebook 13: {unusable}")

### Grid orientation

Checked explicitly rather than assumed. ERA5 latitudes descend while OISST
latitudes ascend, and a silent mismatch would flip a field north-for-south
without any error.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, name in zip(axes, ("sst", "err", "ice"), strict=True):
    field = aux[name].isel(time=0)
    image = ax.pcolormesh(aux["lon"], aux["lat"], field, shading="auto")
    ax.set_title(f"{name} ({OISST_AUX_VARIABLES.get(name, None).units if name in OISST_AUX_VARIABLES else 'Celsius'})")
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
    fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

# Orientation check: latitude must increase upward, longitude rightward, after open_oisst.
print("lat ascending:", bool(np.all(np.diff(aux["lat"].values) > 0)))
print("lon ascending:", bool(np.all(np.diff(aux["lon"].values) > 0)))

### What `err` actually is

OISST's analysis error is a property of the **analysis**, not the ocean. Its
structure follows observation coverage — satellite gaps, buoy density — so a
model that improves when given `err` may be learning where the product is
reliable rather than anything about ocean physics. That is a legitimate
confidence feature, but it is not a physical driver, and it must not be read as
one.

In [ ]:
# The err field is an analysis-uncertainty estimate, so its seasonal and spatial
# structure reflects observation coverage, not ocean dynamics. Worth seeing before
# treating it as a "confidence" input.
err_by_time = np.nanmean(aux["err"].values, axis=(1, 2))
times = aux["time"].values

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(times, err_by_time, linewidth=0.8)
ax.set_ylabel("mean analysis error (C)")
ax.set_title("OISST analysis error over time — driven by observation coverage")
plt.tight_layout()
plt.show()

print(f"err mean {np.nanmean(aux['err'].values):.4f} C, "
      f"std {np.nanstd(aux['err'].values):.4f} C")
print(f"ice: {100 * float(np.nanmean(aux['ice'].values > 0)):.2f}% of cells ever ice-covered")

## ERA5: provenance and licensing

Verified against the live Copernicus CDS catalogue entry:

- **Dataset** — `reanalysis-era5-single-levels`, "ERA5 hourly data on single
  levels from 1940 to present"
- **Licence** — CC-BY-4.0, and the Copernicus licence must be accepted on the
  CDS account before downloads are permitted
- **DOI** — [10.24381/cds.adbb2d47](https://doi.org/10.24381/cds.adbb2d47)
- **Grid** — regular 0.25° lat-lon for the reanalysis
- **Latency** — about 5 days; the recent end of the record is ERA5T, which can
  be revised 2–3 months later

That last point is a **threat to validity for the test period**: an ERA5T value
used as a forecast input today may not be the value stored at the same date
later, so a multivariate result on the most recent months is not exactly
reproducible.

In [ ]:
request = build_era5_request(
    ("u10", "v10", "t2m", "msl", "sshf", "slhf", "ssr"),
    START_DATE,
    END_DATE,
    REGION,
)

print(f"provider : {ERA5_PROVIDER}")
print(f"licence  : {ERA5_LICENSE}")
print(f"DOI      : https://doi.org/{ERA5_DOI}")
print(f"latency  : about {ERA5_LATENCY_DAYS} days; the recent end is ERA5T and may be revised")
print(f"CDS area : {era5_area_from_region(REGION)}  [North, West, South, East]")
print(f"\nvariables requested: {request['request']['variable']}")
print(f"hours per day      : {len(request['request']['time'])}")
print(f"grid               : {request['request']['grid']}")

### Fetching (requires credentials)

Left un-executed so this notebook runs for anyone. The download and alignment
recipe is in the cell below.

In [ ]:
# Submitting the request needs a free Copernicus CDS account and a personal access
# token in ~/.cdsapirc. Left un-executed so this notebook runs without credentials.
#
#     pip install cdsapi
#
#     import cdsapi
#     client = cdsapi.Client()
#     client.retrieve(request["dataset"], request["request"]).download(str(ERA5_PATH))
#
# After downloading, the alignment path is:
#
#     from oisst_fno.multivariate import align_exogenous, daily_from_hourly, variable_spec
#     specs = tuple(variable_spec(name) for name in ("u10", "v10", "t2m"))
#     hourly = xr.open_dataset(ERA5_PATH)
#     daily = daily_from_hourly(hourly, specs)
#     aligned = align_exogenous(daily, aux["sst"], specs)
#
ERA5_PATH = RAW / f"era5_{START_DATE}_{END_DATE}_ne_atlantic.nc"
print("ERA5 target path:", ERA5_PATH)
print("present:", ERA5_PATH.exists())

## Interpretation limits

1. **Predictive is not causal.** If wind channels improve the forecast, that
   shows wind carries information about future SST in this dataset. It does not
   establish that wind drives the change, and nothing here supports an
   attribution claim.
2. **ERA5 is itself a reanalysis**, not observation. Like OISST it is a model
   product, so agreement between them can partly reflect shared assimilation
   inputs rather than independent physical signal.
3. **More channels means more parameters.** Any gain must be shown against the
   matched-budget comparison in notebook `13`, not asserted from a lower loss.

Continue to notebook `13` for the comparison and ablations.